In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import platform
import h5py
# import beautifuljason as bjason

import numpy as np
import pandas as pd

from rdkit import Chem
import pandas as pd
from rdkit.Chem.rdMolDescriptors import CalcMolFormula

In [3]:
# "workingDirectory": {
#     "datatype": "workingDirectory",
#     "count": 1,
#     "data": {
#         "0": "/Users/vsmw51/Library/CloudStorage/Dropbox/Mac/Downloads/4Eric/beta-lapachone"
#     }
# },
# "workingFilename": {
#     "datatype": "workingFilename",
#     "count": 1,
#     "data": {
#         "0": "beta-lapachone"
#     }
# }

def createWorkingDirectory(fn):
    # Create the working directory structure
    working_dir = {
        "datatype": "workingDirectory",
        "count": 1,
        "data": {
            "0": str(fn.parent)
        }
    }

    return working_dir


def createWorkingFilename(fn):
    # Create the working filename structure
    working_filename = {

            "datatype": "workingFilename",
            "count": 1,
            "data": {
                "0": fn.name.split('.')[0]
            }
        }
    
    return working_filename

# createWorkingDirectory(fn), createWorkingFilename(fn)

In [4]:
if platform.system() == 'Windows':
    fn = Path("C:\\Users\\vsmw51\\OneDrive - Durham University\\projects\\programming\\2025\\python\\awh\\bruker_data_sets\\apha_ionone_bruker_jeol\\exam_CMCse_1\\alpha_ionone.jjh5")
elif platform.system() == 'Darwin':
    fn = Path("/opt/topspin4.5.0/examdata/exam_CMCse_1/alpha_ionone.jjh5")
    fn = Path("/Users/vsmw51/OneDrive - Durham University/projects/programming/2025/python/awh/bruker_data_sets/apha_ionone_bruker_jeol/exam_CMCse_1/alpha_ionone.jjh5")

else:
    print("Unsupported OS")
    sys.exit()

In [5]:
fn.name

'alpha_ionone.jjh5'

In [6]:
# get the name of the file and drop the extension
fn.name.split('.')[0]

'alpha_ionone'

In [7]:
# get the the path of parent directory of the file

fn.parent

WindowsPath('C:/Users/vsmw51/OneDrive - Durham University/projects/programming/2025/python/awh/bruker_data_sets/apha_ionone_bruker_jeol/exam_CMCse_1')

In [8]:
def readJEOLmolecule(fn):    
    with h5py.File(fn, 'r') as fp:
        try:
            mol_data = fp['JasonDocument/Molecules/Molecules/0/Atoms']
            print(mol_data.keys())
            atom_list = []
            for i in range(len(mol_data)):
                atom_info = {key: mol_data[str(i)].attrs.get(key, None) for key in mol_data[str(i)].attrs.keys()}
                atom_list.append(atom_info)
            atoms_df = pd.DataFrame(atom_list)
            # add 1 to index
            # atoms_df.index += 1
            # add Z axis all 0
            atoms_df['Z'] = 0.0
        except KeyError as e:
            print(f"KeyError: {e}")

    return atoms_df

def readJEOLmolecule(fn):    
    with h5py.File(fn, 'r') as fp:
        try:
            mol_data = fp['JasonDocument/Molecules/Molecules/0/Atoms']
            print(mol_data.keys())
            atom_list = []
            for atom_idx in mol_data.keys():
                atom_info = {key: mol_data[atom_idx].attrs.get(key, None) for key in mol_data[atom_idx].attrs.keys()}
                atom_list.append(atom_info)
                # add atom_idx to atom_list
                atom_info['atom_idx'] = int(atom_idx) 
            atoms_df = pd.DataFrame(atom_list)
            # add 1 to index
            # atoms_df.index += 1
            # add Z axis all 0
            atoms_df['Z'] = 0.0
        except KeyError as e:
            print(f"KeyError: {e}")

    return atoms_df
atoms_df = readJEOLmolecule(fn)
atoms_df

<KeysViewHDF5 ['0', '1', '10', '11', '12', '13', '2', '3', '4', '5', '6', '7', '8', '9']>


,El,NB.Conn,NB.Num,X,Y,nH,atom_idx,Z
0,6,"[1, 1, 2]","[9, 2, 13]",4.754556,0.182120,0,0,0.0
1,6,"[1, 2]","[5, 2]",2.156452,0.182120,1,1,0.0
2,6,[1],[6],-1.405747,-0.966971,3,10,0.0
3,6,"[513, 513]","[7, 4]",-1.740650,2.432119,2,11,0.0
4,6,[1],[3],2.156452,3.182124,3,12,0.0
5,8,[2],[0],4.754556,-1.317874,0,13,0.0
6,6,"[2, 1]","[1, 0]",3.455554,0.932125,1,2,0.0
7,6,"[1, 513, 514]","[12, 5, 4]",0.857451,2.432119,0,3,0.0
8,6,"[514, 513]","[3, 11]",-0.441555,3.182124,1,4,0.0
9,6,"[513, 1, 513]","[6, 1, 3]",0.857451,0.932125,1,5,0.0


In [9]:
# JasonDocument/Molecules/Molecules/0/NMRData/Spectra/SpectraList/1/Shifts/
# read in predicted chemical shifts from JEOL predictor

    # "c13predictions": {
    #     "datatype": "c13predictions",
    #     "count": 15,
    #     "data": {
    #         "0": {
    #             "atomNumber": 11,
    #             "atom_idx": 1,
    #             "numProtons": 3,
    #             "ppm": 26.743053814773614
    #         },

# print out the C13 shifts for the whole molecule

def extractC13predictionsJEOL(fn:Path, atoms_df:pd.DataFrame)->pd.DataFrame:

    c13shifts_df = pd.DataFrame(columns=['atom_idx',  'atomNumber', 'NumShifts', 'Shifts', 'CalcMethod'])

    with h5py.File(fn, 'r') as f:
        try:
            c13shifts = f['JasonDocument/Molecules/Molecules/0/NMRData/Spectra/SpectraList/0/Shifts']
            
            print('C13 Shifts\n\t', list(c13shifts))
            c13shifts_data = []
            for shift in c13shifts:
                print(shift)
                numC13shifts = len(c13shifts[shift].attrs.get('Value', None))
                calcMethods = c13shifts[shift].attrs.get('Value.Method', None)
                # print(f"  {int(shift)+1}, \t{numC13shifts},\t{int(c13shifts[shift].attrs.get('Nums', None)[0])+1}:\t{c13shifts[shift].attrs.get('Value', None)}\t{calcMethods}")
                # add row to dataframe
                c13_row = {
                    'atom_idx': int(c13shifts[shift].attrs.get('Nums', None)[0]),
                    'atomNumber': int(c13shifts[shift].attrs.get('Nums', None)[0])+1,
                    'NumShifts': numC13shifts,
                    'Shifts': c13shifts[shift].attrs.get('Value', None),
                    'CalcMethod': calcMethods
                }
                c13shifts_data.append(c13_row)


            c13shifts_df = pd.DataFrame(c13shifts_data)

            for row in atoms_df.itertuples():
                atom_idx = row.atom_idx
                numProtons = int(row.nH)
                # replace numProtons in c13shifts_df using atom_idx
                if atom_idx in c13shifts_df.index.values:
                    c13shifts_df.loc[c13shifts_df.atom_idx == atom_idx, "numProtons"] = numProtons
            #  convert numProtons to integer
            c13shifts_df["numProtons"] = c13shifts_df["numProtons"].astype(int)
        except KeyError as e:
            print(f"KeyError: {e}")



    

    # sort dataframe by atom index
    # c13shifts_df = c13shifts_df.sort_values(by='atom_idx')

    return c13shifts_df

In [10]:
['0', '1', '10', '11', '12', '13', '2', '3', '4', '5', '6', '7', '8', '9']

['0', '1', '10', '11', '12', '13', '2', '3', '4', '5', '6', '7', '8', '9']

In [11]:
c13shifts_df = extractC13predictionsJEOL(fn, atoms_df)
c13shifts_df

C13 Shifts
	 ['0', '1', '10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9']
0
1
10
11
12
2
3
4
5
6
7
8
9


,atom_idx,atomNumber,NumShifts,Shifts,CalcMethod,numProtons
0,0,1,4,"[198.3108, 198.01193, 197.2, 197.2]","[1, 2, 4, 5]",0
1,1,2,4,"[148.92451, 147.93091, 148.25, 148.25]","[1, 2, 4, 5]",1
2,10,11,3,"[26.21915, 27.36111, 27.36111]","[2, 4, 5]",3
3,11,12,3,"[22.827124, 23.044445, 23.044445]","[2, 4, 5]",2
4,12,13,3,"[22.699148, 22.862501, 22.862501]","[2, 4, 5]",3
5,4,5,4,"[122.59451, 122.61086, 122.55, 122.55]","[1, 2, 4, 5]",1
6,5,6,4,"[54.24827, 56.50496, 54.25, 54.25]","[1, 2, 4, 5]",1
7,2,3,3,"[133.10081, 132.45, 132.45]","[2, 4, 5]",1
8,3,4,3,"[132.65195, 132.05, 132.05]","[2, 4, 5]",0
9,6,7,3,"[34.327927, 32.4, 32.4]","[2, 4, 5]",0


In [27]:
# add numProtons using atoms_df dataframe


# for row in atoms_df.itertuples():
#     atom_idx = row.Index
#     numProtons = int(row.nH)
#     # replace numProtons in c13shifts_df using atom_idx
#     if atom_idx in c13shifts_df.atom_idx.values:
#         c13shifts_df.loc[c13shifts_df.atom_idx == atom_idx, "numProtons"] = numProtons

for row in atoms_df.itertuples():
    atom_idx = row.atom_idx
    numProtons = int(row.nH)
    # replace numProtons in c13shifts_df using atom_idx
    if atom_idx in c13shifts_df.index.values:
        c13shifts_df.loc[c13shifts_df.index == atom_idx, "numProtons"] = numProtons

#  convert numProtons to integer
c13shifts_df["numProtons"] = c13shifts_df["numProtons"].astype(int)
c13shifts_df

,atom_idx,atomNumber,NumShifts,Shifts,CalcMethod,numProtons
0,0,1,4,"[198.3108, 198.01193, 197.2, 197.2]","[1, 2, 4, 5]",0
1,1,2,4,"[148.92451, 147.93091, 148.25, 148.25]","[1, 2, 4, 5]",1
2,10,11,3,"[26.21915, 27.36111, 27.36111]","[2, 4, 5]",1
3,11,12,3,"[22.827124, 23.044445, 23.044445]","[2, 4, 5]",0
4,12,13,3,"[22.699148, 22.862501, 22.862501]","[2, 4, 5]",1
5,2,5,4,"[122.59451, 122.61086, 122.55, 122.55]","[1, 2, 4, 5]",1
6,3,6,4,"[54.24827, 56.50496, 54.25, 54.25]","[1, 2, 4, 5]",0
7,4,3,3,"[133.10081, 132.45, 132.45]","[2, 4, 5]",2
8,5,4,3,"[132.65195, 132.05, 132.05]","[2, 4, 5]",3
9,6,7,3,"[34.327927, 32.4, 32.4]","[2, 4, 5]",3


In [14]:
atoms_df

,El,NB.Conn,NB.Num,X,Y,nH,Z
0,6,"[1, 1, 2]","[9, 2, 13]",4.754556,0.182120,0,0.0
1,6,"[1, 2]","[5, 2]",2.156452,0.182120,1,0.0
2,6,"[2, 1]","[1, 0]",3.455554,0.932125,1,0.0
3,6,"[1, 513, 514]","[12, 5, 4]",0.857451,2.432119,0,0.0
4,6,"[514, 513]","[3, 11]",-0.441555,3.182124,1,0.0
5,6,"[513, 1, 513]","[6, 1, 3]",0.857451,0.932125,1,0.0
6,6,"[1, 1, 513, 513]","[10, 8, 7, 5]",-0.441555,0.182120,0,0.0
7,6,"[513, 513]","[6, 11]",-1.740650,0.932125,2,0.0
8,6,[1],[6],0.522561,-0.966971,3,0.0
9,6,[1],[0],6.053656,0.932125,3,0.0


In [16]:
c13predictions = {
    "datatype": "c13predictions",
    "count": len(c13shifts_df),
    "data": {}
}

for idx, row in c13shifts_df.iterrows():
    c13predictions["data"][row["atom_idx"]] = {
        "atom_idx": row["atom_idx"],
        "atomNumber": row["atomNumber"],
        "numProtons": row["numProtons"],
        "ppm": float(row["Shifts"][-1])
    }
c13predictions

{'datatype': 'c13predictions',
 'count': 13,
 'data': {0: {'atom_idx': 0,
   'atomNumber': 1,
   'numProtons': 0,
   'ppm': 197.1999969482422},
  1: {'atom_idx': 1, 'atomNumber': 2, 'numProtons': 1, 'ppm': 148.25},
  2: {'atom_idx': 2,
   'atomNumber': 5,
   'numProtons': 1,
   'ppm': 122.55000305175781},
  3: {'atom_idx': 3, 'atomNumber': 6, 'numProtons': 0, 'ppm': 54.25},
  4: {'atom_idx': 4,
   'atomNumber': 3,
   'numProtons': 1,
   'ppm': 132.4499969482422},
  5: {'atom_idx': 5,
   'atomNumber': 4,
   'numProtons': 1,
   'ppm': 132.0500030517578},
  6: {'atom_idx': 6,
   'atomNumber': 7,
   'numProtons': 0,
   'ppm': 32.400001525878906},
  7: {'atom_idx': 7,
   'atomNumber': 8,
   'numProtons': 2,
   'ppm': 29.900001525878906},
  8: {'atom_idx': 8,
   'atomNumber': 9,
   'numProtons': 3,
   'ppm': 27.36111068725586},
  9: {'atom_idx': 9,
   'atomNumber': 10,
   'numProtons': 3,
   'ppm': 27.100000381469727},
  10: {'atom_idx': 10,
   'atomNumber': 11,
   'numProtons': 3,
   'ppm':

In [18]:
[ shift[-1] for shift in c13shifts_df.loc[:,"Shifts"] ]


[np.float32(197.2),
 np.float32(148.25),
 np.float32(27.36111),
 np.float32(23.044445),
 np.float32(22.862501),
 np.float32(122.55),
 np.float32(54.25),
 np.float32(132.45),
 np.float32(132.05),
 np.float32(32.4),
 np.float32(29.900002),
 np.float32(27.36111),
 np.float32(27.1)]

In [1]:
from qtpy.QtWidgets import QProgressDialog

In [2]:
QProgressDialog?

Init signature: QProgressDialog(self, /, *args, **kwargs)
Docstring:     
QProgressDialog(parent: Optional[QWidget] = None, flags: Union[Qt.WindowFlags, Qt.WindowType] = Qt.WindowFlags())
QProgressDialog(labelText: Optional[str], cancelButtonText: Optional[str], minimum: int, maximum: int, parent: Optional[QWidget] = None, flags: Union[Qt.WindowFlags, Qt.WindowType] = Qt.WindowFlags())
File:           ~/opt/anaconda3/envs/py313/lib/python3.13/site-packages/PyQt5/QtWidgets.abi3.so
Type:           wrappertype
Subclasses:     

In [16]:
fn = Path("/Users/vsmw51/OneDrive - Durham University/projects/programming/2025/python/awh/bruker_data_sets/apha_ionone_bruker_jeol/exam_CMCse_1/alpha_ionone.jjh5")
fn.exists()

True